**Real-Agent Causal Recovery.** A live DeepSeek coding agent inspects the AgentTX ledger, selects an injected faulty producer, invokes causal rollback, and finishes the task while preserving independent work. The per-repeat panels expose latency and decision variability; the final panel reports the semantic invariants that must all hold. This is a decision-layer validation, not a comparison against a scripted baseline.

In [ ]:
# ipython -c "%run plot_real_agent_recovery.ipynb"

import ast
import os
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path

STANDARD_WIDTH = 17.8

def cm_to_inch(value):
    return value / 2.54

plt.rcParams.update(plt.rcParamsDefault)
matplotlib.rcParams['text.usetex'] = False
plt.rcParams['font.family'] = 'Nimbus Roman'
plt.rcParams['axes.grid'] = False
plt.rcParams['axes.linewidth'] = 0.6
plt.rcParams['xtick.direction'] = 'in'
plt.rcParams['ytick.direction'] = 'in'
plt.rcParams['legend.frameon'] = True
plt.rcParams['legend.edgecolor'] = '0.55'
plt.rcParams['legend.framealpha'] = 1.0
plt.rcParams['legend.fancybox'] = False

OURS = dict(color='#c00000', marker='s', linestyle='-', linewidth=1.0, markersize=3.2)
AUX = dict(color='#4f9fcf', marker='^', linestyle='-.', linewidth=0.9, markersize=3.2, markerfacecolor='none')
LEDGER = dict(color='black', marker='o', linestyle='--', linewidth=0.8, markersize=2.8, markerfacecolor='none')

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'
FIGDIR = ROOT / 'motivation'

def provider_result(name):
    """Prefer the default provider's result dir, then any provider dir,
    then the legacy top-level result file."""
    results = ROOT / 'experiments' / 'results'
    provider = os.environ.get('AGENTTX_PROVIDER', 'deepseek')
    preferred = results / provider / name
    if preferred.exists():
        return preferred
    directories = sorted(
        path for path in results.iterdir()
        if path.is_dir() and (path / name).exists()
    )
    if directories:
        return directories[0] / name
    return results / name
df = pd.read_csv(provider_result('real_agent_recovery.csv')).sort_values('repeat')
repeats = df['repeat'].to_numpy(dtype=int) + 1

fig = plt.figure(dpi=300, figsize=(cm_to_inch(STANDARD_WIDTH), cm_to_inch(7.0)))

ax = plt.subplot(2, 2, 1)
latency, = ax.plot(repeats, df['wall_s'], **OURS, label='end-to-end latency')
ax.axhline(df['wall_s'].median(), color='0.45', linestyle=':', linewidth=0.7, label='median')
ax.set_ylabel('Task latency (s)', fontsize=8)
ax.set_xlabel('Agent run\n(a) Live recovery latency', fontsize=7)
ax.set_xticks(repeats)

ax = plt.subplot(2, 2, 2)
tools, = ax.plot(repeats, df['tool_calls'], **AUX, label='tool calls')
ledger, = ax.plot(repeats, df['ledger_steps'], **LEDGER, label='ledger steps')
ax.set_ylabel('Count', fontsize=8)
ax.set_xlabel('Agent run\n(b) Trajectory size', fontsize=7)
ax.set_xticks(repeats)
ax.legend(loc='upper center', fontsize=6.5, ncol=2, handlelength=1.5, borderpad=0.25)

ax = plt.subplot(2, 2, 3)
targets = df['rollback_targets'].map(lambda value: len(ast.literal_eval(value)))
rollback, = ax.plot(repeats, targets, **OURS, label='invalidated steps')
inspect, = ax.plot(repeats, df['inspect_calls'], **AUX, label='ledger inspections')
ax.set_ylabel('Count', fontsize=8)
ax.set_xlabel('Agent run\n(c) Recovery decision', fontsize=7)
ax.set_xticks(repeats)
ax.legend(loc='upper center', fontsize=6.5, ncol=2, handlelength=1.5, borderpad=0.25)

ax = plt.subplot(2, 2, 4)
checks = [
    ('Root', 'selected_root'), ('Targets', 'causal_targets_correct'),
    ('Retain', 'independent_retained'), ('Remove', 'derived_removed'),
    ('Tests', 'success'), ('No leak', 'host_polluted_before_commit'),
]
rates = []
for label, column in checks:
    values = df[column].astype(str).str.lower().map({'true': 1.0, 'false': 0.0})
    rates.append(100.0 * ((1.0 - values.mean()) if label == 'No leak' else values.mean()))
semantic, = ax.plot(range(len(checks)), rates, **OURS, label='AgentTX + live agent')
ax.set_ylabel('Runs satisfying invariant (%)', fontsize=8)
ax.set_xlabel('Recovery property\n(d) Semantic correctness', fontsize=7)
ax.set_xticks(range(len(checks)), [item[0] for item in checks], rotation=20, ha='right')
ax.set_ylim(-5, 105)

for ax in fig.axes:
    ax.tick_params(axis='both', labelsize=7)

fig.legend(handles=[latency, tools, ledger, rollback, inspect, semantic],
           loc='upper center', bbox_to_anchor=(0.5, 1.035), ncol=6,
           fontsize=6.1, columnspacing=0.7, handlelength=1.5, handletextpad=0.3, borderpad=0.3)
plt.tight_layout(pad=0.6, h_pad=1.6, w_pad=1.2, rect=[0.0, 0.0, 1.0, 0.91])
plt.savefig(FIGDIR / 'FIG-Real-Agent-Recovery.pdf', bbox_inches='tight', pad_inches=0.02,
            metadata={'CreationDate': None, 'ModDate': None})
plt.savefig(FIGDIR / 'FIG-Real-Agent-Recovery.png', dpi=300, bbox_inches='tight', pad_inches=0.02)
plt.show()

print(f"latency p50={df['wall_s'].median():.2f}s; success={df['success'].astype(str).str.lower().eq('true').mean():.0%}; runs={len(df)}")
